#### **Serialization, Configuration & Strict Mode**

#### **Serialization**

**Serialization** means converting a Pydantic model into data formats such as a Python dictionary or JSON. Suppose:

In [1]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int
    email: str

user = User(
    name='Shourov Roy',
    age=23,
    email="shourov@example.com"
)

**`model_dump()`:** The main Pydantic v2 method for converting a model to a dictionary is:

In [2]:
data = user.model_dump()
print(data)

{'name': 'Shourov Roy', 'age': 23, 'email': 'shourov@example.com'}


---
**`model_dump_json()`:** If you want JSON:

In [3]:
json_data = user.model_dump_json()
print(json_data)

{"name":"Shourov Roy","age":23,"email":"shourov@example.com"}


---

#### **Serialization of Nested Models**

In [5]:
from pydantic import BaseModel

class Address(BaseModel):
    city: str
    country: str

class User(BaseModel):
    name: str
    address: Address

user = User(
    name="Shourov Roy",
    address={
        "city": "Dinajpur",
        "country": "Bangladesh"
    }
)
print(user.model_dump())

{'name': 'Shourov Roy', 'address': {'city': 'Dinajpur', 'country': 'Bangladesh'}}


---

#### **Serialization Options**

**Exclude unset fields**

```python
user.model_dump(exclude_unset=True)
```

This is useful when you only want fields explicitly provided by the user.

---
**Exclude `None`**

```python
user.model_dump(exclude_none=True)
```
Example:

```python
class User(BaseModel):
    name: str
    nickname: str | None = None

user = User(name="Shourov")
print(user.model_dump(exclude_none=True))
```
---
**Include specific fields**

```python
user.model_dump(include={"name", "email"})
```

---

**Exclude fields**

```python
user.model_dump(exclude={"email"})
```

---

#### **Configuration**

Pydantic allows you to control model behavior using `ConfigDict`. In Pydantic v2:

In [6]:
from pydantic import BaseModel, ConfigDict

class User(BaseModel):
    model_config = ConfigDict(
        str_strip_whitespace=True
    )
    name: str

user = User(name="   Shourov   ")
print(user.name)

Shourov


Pydantic removed the surrounding whitespace.

---

#### **Common Configuration Options**

**`str_strip_whitespace`:** Remove whitespace from strings.

```python
model_config = ConfigDict(
    str_strip_whitespace=True
)
```

---
**`str_to_lower`:** Convert strings to lowercase.

```python
model_config = ConfigDict(
    str_to_lower=True
)
```
---
**`str_to_upper`:** Convert strings to uppercase.

```python
model_config = ConfigDict(
    str_to_upper=True
)
```

---
**Extra Fields:** Suppose your model is:

```python
class User(BaseModel):
    name: str
    age: int
```
But the input contains:

```python
{
    "name": "Shourov",
    "age": 25,
    "country": "Bangladesh"
}
```

What should happen to `country`? Pydantic lets you configure this. Ignore extra fields.

```python
class User(BaseModel):
    model_config = ConfigDict(
        extra="ignore"
    )

    name: str
    age: int
```

`country` will be ignored.

---
**Forbid extra fields**

```python
class User(BaseModel):
    model_config = ConfigDict(
        extra="forbid"
    )
    name: str
    age: int

User(
    name="Shourov",
    age=25,
    country="Bangladesh"
)
```

causes a validation error because `country` isn't allowed.

---
**Allow extra fields**

```python
class User(BaseModel):
    model_config = ConfigDict(
        extra="allow"
    )
    name: str
    age: int
```

Additional fields can be accepted and stored.

---

#### **Strict Mode**

By default, Pydantic is often **flexible** with compatible input types. For example:

In [1]:
from pydantic import BaseModel

class User(BaseModel):
    age: int

# This may work
user = User(age='25')
print(user.age)
print(type(user.age))

25
<class 'int'>


This behavior is called **lax/coercive validation**.

---

Strict mode tells Pydantic:

> "Don't automatically convert values into the expected type."

In [2]:
from pydantic import BaseModel, ConfigDict

class User(BaseModel):
    model_config = ConfigDict(
        strict = True
    )
    age: int

# user = User(age='25') this is not work here
user = User(age=25)
print(user.age)

25


Strict mode requires the correct type rather than coercing the string to an integer.

---
You don't always need strict mode for the entire model. You can use `StrictInt`, `StrictStr`, etc.

```python
from pydantic import BaseModel, StrictInt

class User(BaseModel):
    age: StrictInt
```

---
**Using `Strict()`:** Pydantic also provides `Strict()` through `Annotated`.

```python
from typing import Annotated
from pydantic import BaseModel, Strict

class User(BaseModel):
    age: Annotated[int, Strict()]
```

Now `age` must be an actual integer.

---

#### **Quick Cheat Sheet**

| Concept                  | Main tool                   | Purpose                         |
| ------------------------ | --------------------------- | ------------------------------- |
| Nested Models            | `class Address(BaseModel)`  | Model complex structures        |
| Dictionary serialization | `model_dump()`              | Model → `dict`                  |
| JSON serialization       | `model_dump_json()`         | Model → JSON                    |
| Configuration            | `ConfigDict()`              | Control model behavior          |
| Extra fields             | `extra="forbid"`            | Reject unknown fields           |
| Whitespace               | `str_strip_whitespace=True` | Clean strings                   |
| Strict model             | `strict=True`               | Disable type coercion           |
| Strict field             | `StrictInt`                 | Strict validation for one field |
| Strict annotation        | `Annotated[int, Strict()]`  | Field-level strictness          |